# Step 1 — PlantSeg: download, explore, choose a crop

**Project:** Field-robust disease severity estimation for low-end smartphones

Goals of this notebook:
1. Download and verify PlantSeg (latest Zenodo release)
2. Understand its structure and metadata
3. Count images per plant and disease, and shortlist crops relevant to Indian farmers
4. Look at lesion-area ratios and image resolutions
5. Visualise image + mask pairs
6. Save a subset CSV for the chosen crop (used by the training notebook)

Runtime: CPU is fine for this notebook. Dataset licence: CC BY-NC-ND 4.0 (non-commercial, don't redistribute modified annotations).

In [ ]:
import os, hashlib, zipfile, subprocess, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

DATA_ROOT = Path("/content/data")   # change if running locally
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Latest PlantSeg release on Zenodo (v6, July 2025). If the direct link fails,
# download Plantseg.zip manually from https://zenodo.org/records/16665770 and
# put it at ZIP_PATH.
ZIP_URL  = "https://zenodo.org/records/16665770/files/Plantseg.zip?download=1"
ZIP_PATH = DATA_ROOT / "Plantseg.zip"
EXPECTED_MD5 = "23385d877bc84bab51b54630055c75b8"

random.seed(42); np.random.seed(42)

## 1. Download and verify

In [ ]:
def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

if not ZIP_PATH.exists():
    print("Downloading (~1.7 GB)...")
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(ZIP_PATH), ZIP_URL], check=True)

checksum = md5sum(ZIP_PATH)
print("MD5 OK" if checksum == EXPECTED_MD5 else f"MD5 mismatch: {checksum} (re-download)")

EXTRACT_DIR = DATA_ROOT / "PlantSeg"
if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
print("Extracted to", EXTRACT_DIR)

## 2. Inspect the folder structure
Folder names can change between releases, so we discover them instead of hard-coding.

In [ ]:
def show_tree(path, max_depth=3, max_items=8, depth=0):
    if depth > max_depth:
        return
    items = sorted(path.iterdir())
    for p in items[:max_items]:
        n = f" ({len(list(p.iterdir()))} items)" if p.is_dir() else ""
        print("  " * depth + ("📁 " if p.is_dir() else "📄 ") + p.name + n)
        if p.is_dir():
            show_tree(p, max_depth, max_items, depth + 1)
    if len(items) > max_items:
        print("  " * depth + f"... +{len(items) - max_items} more")

show_tree(EXTRACT_DIR)

# Index every file by name so relative paths in the metadata can always be resolved
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
all_files = [p for p in EXTRACT_DIR.rglob("*") if p.is_file()]
by_name = {}
for p in all_files:
    by_name.setdefault(p.name, []).append(p)
print(f"\n{len(all_files):,} files, {sum(p.suffix.lower() in IMG_EXT for p in all_files):,} images")

## 3. Load the metadata

In [ ]:
meta_files = [p for p in all_files if p.suffix.lower() == ".csv" and "meta" in p.name.lower()]
print("Metadata candidates:", meta_files)
assert meta_files, "No metadata CSV found - check the tree above"

meta = pd.read_csv(meta_files[0])
meta.columns = [c.strip().lower().replace(" ", "_") for c in meta.columns]
print(meta.shape)
print(meta.columns.tolist())
meta.head()

In [ ]:
def find_col(df, *keywords, exclude=()):
    """First column whose name contains all keywords (and none of exclude)."""
    for c in df.columns:
        if all(k in c for k in keywords) and not any(e in c for e in exclude):
            return c
    return None

COL = {
    "image":   find_col(meta, "name") or find_col(meta, "image"),
    "plant":   find_col(meta, "plant"),
    "disease": find_col(meta, "disease"),
    "label":   find_col(meta, "label") or find_col(meta, "mask", exclude=("ratio",)),
    "ratio":   find_col(meta, "ratio"),
    "res":     find_col(meta, "resolution") or find_col(meta, "size"),
    "split":   find_col(meta, "split") or find_col(meta, "subset"),
}
COL
# If any value is None, set it by hand from the column list printed above.

## 4. Images per plant type

In [ ]:
plant_counts = meta[COL["plant"]].value_counts()
print(f"{meta[COL['plant']].nunique()} plant types, {meta[COL['disease']].nunique()} diseases")

plt.figure(figsize=(10, 9))
plant_counts.sort_values().plot.barh(color="seagreen")
plt.xlabel("diseased images"); plt.title("PlantSeg: images per plant type")
plt.tight_layout(); plt.show()

## 5. Shortlist crops that matter to Indian farmers
Edit the keyword list freely.

In [ ]:
INDIA_CROPS = ["tomato", "pepper", "chili", "chilli", "capsicum", "rice", "potato", "wheat",
               "maize", "corn", "cotton", "soybean", "banana", "mango", "grape", "onion",
               "sugarcane", "groundnut", "peanut", "citrus", "orange", "cucumber", "eggplant", "brinjal"]

plant_lower = meta[COL["plant"]].astype(str).str.lower()
rows = []
for plant in meta[COL["plant"]].unique():
    p = str(plant).lower()
    if any(k in p for k in INDIA_CROPS):
        sub = meta[plant_lower == p]
        row = {"plant": plant, "images": len(sub), "diseases": sub[COL["disease"]].nunique(),
               "min_imgs_per_disease": sub[COL["disease"]].value_counts().min()}
        if COL["split"]:
            row.update(sub[COL["split"]].value_counts().to_dict())
        rows.append(row)

shortlist = pd.DataFrame(rows).sort_values("images", ascending=False).reset_index(drop=True)
shortlist

**How to choose:** prefer a crop with (a) the most images, (b) at least ~80–100 images per disease you keep, and (c) real importance in your region. You can drop rare diseases rather than dropping the crop.

In [ ]:
CROP = shortlist.loc[0, "plant"]   # <- replace with your choice, e.g. "tomato"
crop_df = meta[plant_lower == str(CROP).lower()].copy()

print(f"{CROP}: {len(crop_df)} images")
crop_df[COL["disease"]].value_counts().plot.barh(figsize=(8, 4), color="indianred")
plt.title(f"{CROP}: images per disease"); plt.tight_layout(); plt.show()

## 6. Lesion-area ratio

The mask ratio is **diseased pixels ÷ whole-image pixels**. It is *not* severity yet: severity needs the leaf area as the denominator, and PlantSeg has no leaf masks. That's the gap Step 3 fills.

In [ ]:
def resolve(p):
    """Turn a metadata path/filename into a real file path."""
    if pd.isna(p):
        return None
    p = Path(str(p))
    for cand in (EXTRACT_DIR / p, EXTRACT_DIR / p.name):
        if cand.exists():
            return cand
    hits = by_name.get(p.name)
    return hits[0] if hits else None

def mask_path_for(row):
    if COL["label"] and resolve(row[COL["label"]]):
        return resolve(row[COL["label"]])
    stem = Path(str(row[COL["image"]])).stem   # fall back: same stem, .png
    hits = by_name.get(stem + ".png", [])
    hits = [h for h in hits if "annot" in str(h).lower() or "label" in str(h).lower() or "mask" in str(h).lower()]
    return hits[0] if hits else None

if COL["ratio"]:
    ratios = pd.to_numeric(crop_df[COL["ratio"]], errors="coerce")
else:   # compute from masks on a sample
    sample = crop_df.sample(min(300, len(crop_df)), random_state=0)
    vals = []
    for _, r in sample.iterrows():
        mp = mask_path_for(r)
        if mp:
            m = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
            vals.append((m > 0).mean())
    ratios = pd.Series(vals)

print(ratios.describe())
ratios.plot.hist(bins=40, color="darkorange", figsize=(7, 3.5))
plt.xlabel("diseased pixels / image pixels"); plt.title(f"{CROP}: lesion-area ratio")
plt.tight_layout(); plt.show()

## 7. Look at image + mask pairs
Check: are these close-ups or whole plants? Busy backgrounds? Several leaves per photo?

In [ ]:
def overlay(img, mask, alpha=0.45):
    out = img.copy()
    red = np.zeros_like(img); red[..., 0] = 255
    m = mask > 0
    out[m] = (alpha * red[m] + (1 - alpha) * img[m]).astype(np.uint8)
    return out

samples = crop_df.sample(min(6, len(crop_df)), random_state=1)
fig, axes = plt.subplots(len(samples), 2, figsize=(9, 3.2 * len(samples)))
axes = np.atleast_2d(axes)
for ax_row, (_, r) in zip(axes, samples.iterrows()):
    ip, mp = resolve(r[COL["image"]]), mask_path_for(r)
    if ip is None or mp is None:
        for a in ax_row: a.axis("off")
        continue
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
    if mask.shape[:2] != img.shape[:2]:
        mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    ax_row[0].imshow(img); ax_row[0].set_title(r[COL["disease"]], fontsize=9)
    ax_row[1].imshow(overlay(img, mask)); ax_row[1].set_title(f"lesion ratio {(mask > 0).mean():.2%}", fontsize=9)
    for a in ax_row: a.axis("off")
plt.tight_layout(); plt.show()

## 8. Image resolution
This tells us what input size is realistic, and how far these photos are from a cheap phone's.

In [ ]:
dims = []
for _, r in crop_df.sample(min(300, len(crop_df)), random_state=2).iterrows():
    ip = resolve(r[COL["image"]])
    if ip:
        h, w = cv2.imread(str(ip)).shape[:2]
        dims.append((w, h))
dims = pd.DataFrame(dims, columns=["width", "height"])
print(dims.describe().round(0))
plt.scatter(dims.width, dims.height, s=8, alpha=0.5)
plt.xlabel("width"); plt.ylabel("height"); plt.title(f"{CROP}: image sizes"); plt.show()

## 9. Save the crop subset for training

In [ ]:
crop_df["image_path"] = [str(resolve(p)) if resolve(p) else None for p in crop_df[COL["image"]]]
crop_df["mask_path"]  = [str(m) if (m := mask_path_for(r)) else None for _, r in crop_df.iterrows()]
missing = crop_df[["image_path", "mask_path"]].isna().any(axis=1).sum()
print(f"Unresolved rows: {missing}")

out = DATA_ROOT / f"subset_{str(CROP).lower().replace(' ', '_')}.csv"
crop_df.dropna(subset=["image_path", "mask_path"]).to_csv(out, index=False)
print("Saved", out)

## What to note down before Step 2
- Chosen crop, and which diseases you keep (and why)
- Train / val / test counts
- Typical lesion ratio and image size
- Your visual impression: close-ups vs whole plants, background clutter

**Next:** Step 2 — baseline lightweight segmentation model (SegFormer-B0 / MobileNet DeepLabV3+) on this subset, reporting mIoU and lesion-area error.